# 第九章：转录因子调控网络（decoupler + CollecTRI）

**scRNA-seq 实践教程系列** | 基于 GSE136103 肝硬化数据集

---

## 写在前面

上一篇我们分析了细胞之间的对话——谁在和谁说话。这一篇，我们把视角转向细胞内部：在每种细胞类型中，是哪些转录因子（Transcription Factor, TF）在"指挥"基因表达程序？

差异表达分析告诉我们"基因 X 在疾病中上调了"。但基因不会自发地上调——它的启动子区域被特定的转录因子结合、激活。找到这些"幕后指挥"，才是理解基因调控机制的关键。

比如：如果我们发现肝硬化中巨噬细胞的 STAT1 活性显著升高，那就不只是"一堆炎症基因上调了"——而是"STAT1 驱动的干扰素应答程序被激活了"。这提供了一个更高层次的机制解释，也指向了可能的干预靶点。

这一篇，我们用 decoupler + CollecTRI 来推断转录因子活性。

## 为什么选 decoupler + CollecTRI

### 从基因表达推断 TF 活性

直接测量 TF 活性在技术上很困难——TF 的 mRNA 表达量和它的实际功能活性之间的相关性很弱。一个 TF 可能高表达但没有被磷酸化激活，也可能低表达但通过翻译后修饰处于高活性状态。

更好的策略是"看结果推原因"：如果一个 TF 的靶基因们（target genes）整体上调了，那这个 TF 大概率是被激活的——即使它自己的 mRNA 水平没有变化。

这就是 decoupler 的思路（Badia-i-Mompel et al., Bioinformatics Advances, 2022）。

### CollecTRI：TF-靶基因数据库

要从靶基因推断 TF 活性，首先需要一个可靠的 TF-靶基因关系数据库。CollecTRI（Müller-Dott et al., 2023）整合了 12 个来源的实验验证 TF-靶基因关系，质量优于 DoRothEA 等早期数据库。

### ULM：单变量线性模型

decoupler 提供了多种推断方法。我们使用 ULM（Univariate Linear Model）——对每个 TF，用它的靶基因表达值做单变量线性回归，回归系数的 t-statistic 作为 TF 活性分数。

ULM 的优势是简洁、快速、可解释性强。对于 scRNA-seq 数据中常见的稀疏矩阵，ULM 比多变量方法更稳健。

## Part A：TF 活性推断

### 加载数据和 CollecTRI 网络

In [ ]:
import scanpy as sc
import decoupler as dc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

adata = sc.read_h5ad("results/04_annotated.h5ad")
print(f"数据维度: {adata.shape}")

# 获取 CollecTRI 网络
net = dc.op.collectri()
print(f"CollecTRI 网络:")
print(f"  交互数: {len(net)}")
print(f"  TF 数: {net['source'].nunique()}")

**输出：**

> 📊 输出：
> 数据维度: (58735, 28553)
> CollecTRI 网络:
>   交互数: 42990
>   TF 数: 1185

CollecTRI 包含 42,990 个 TF-靶基因交互关系，覆盖 1,185 个转录因子。每条记录包含三列：source（TF）、target（靶基因）、weight（激活=+1 / 抑制=-1）。

⚠️ 踩坑预警：decoupler 2.x 的 API 大改

如果你之前用过 decoupler 1.x，请注意 2.x 完全重构了 API：

操作
旧版 (1.x)
新版 (2.x)

获取 CollecTRI
dc.get_collectri()
dc.op.collectri()

运行 ULM
dc.run_ulm(adata, net, source='source', target='target', weight='weight')
dc.mt.ulm(adata, net)

结果存储位置
adata.obsm['ulm_estimate']
adata.obsm['score_ulm']

获取 progeny
dc.get_progeny()
dc.op.progeny()

2.x 版本把所有方法放到了 dc.mt 子模块下，数据库获取放到了 dc.op 子模块下。参数也简化了——不再需要手动指定 source/target/weight 列名，decoupler 自动识别。

网上大量教程还在用旧版 API。 如果你看到 dc.run_ulm() 的代码，那是旧版的。直接复制到 2.x 环境里会报 AttributeError: module 'decoupler' has no attribute 'run_ulm'。

### 运行 ULM

In [ ]:
dc.mt.ulm(
    adata,
    net=net,
    use_raw=True,
    verbose=True
)

print("结果存储位置:")
print(f"  TF 活性分数: adata.obsm['score_ulm']"
      f" {adata.obsm['score_ulm'].shape}")
print(f"  p-value: adata.obsm['pval_ulm']"
      f" {adata.obsm['pval_ulm'].shape}")

**输出：**

> 📊 输出：
> 结果存储位置:
>   TF 活性分数: adata.obsm['score_ulm'] (58735, 1185)
>   p-value: adata.obsm['pval_ulm'] (58735, 1185)

ULM 为每个细胞（58,735）× 每个 TF（1,185）计算了一个活性分数。正值表示该 TF 在这个细胞中是激活的（靶基因整体上调），负值表示抑制的。

### 计算每种细胞类型的平均 TF 活性

单细胞层面的 TF 活性噪声很大。我们先聚合到细胞类型水平：

In [ ]:
# 按细胞类型取平均
tf_scores = adata.obsm["score_ulm"].copy()
tf_scores["cell_type"] = adata.obs["cell_type"].values
tf_mean = tf_scores.groupby("cell_type").mean()

# 保存
tf_mean.to_csv(
    "results/tables/tf_activity_mean_celltype.csv"
)
print(f"✅ 已保存: tf_activity_mean_celltype.csv"
      f" ({tf_mean.shape})")

**输出：**

> 📊 输出：
> ✅ 已保存: tf_activity_mean_celltype.csv (12, 1185)

### 关键 TF 热图

In [ ]:
import matplotlib.pyplot as plt

# 挑选生物学关键 TF
key_tfs = [
    # 髓系/巨噬细胞
    "SPI1", "CEBPB", "MAFB", "MAF",
    # 肝细胞
    "HNF4A", "HNF1A", "CEBPA",
    # 炎症/免疫
    "STAT1", "STAT3", "IRF1", "IRF7", "NFKB1",
    # T 细胞
    "TBX21", "EOMES", "GATA3",
    # Treg
    "FOXP3",
    # B 细胞
    "PAX5", "EBF1",
    # 纤维化
    "SMAD3", "TWIST1",
]
key_tfs = [tf for tf in key_tfs
           if tf in tf_mean.columns]

fig, ax = plt.subplots(figsize=(14, 8))
data = tf_mean[key_tfs].T
im = ax.imshow(data.values, cmap="RdBu_r",
               aspect="auto", vmin=-3, vmax=3)
ax.set_xticks(range(len(data.columns)))
ax.set_xticklabels(data.columns, rotation=45,
                    ha="right", fontsize=9)
ax.set_yticks(range(len(data.index)))
ax.set_yticklabels(data.index, fontsize=9)
ax.set_title("TF Activity by Cell Type (ULM)")
plt.colorbar(im, ax=ax, shrink=0.8, label="TF score")
plt.savefig(
    "results/figures/08_tf_activity_heatmap.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 1：各细胞类型的转录因子活性热图

图 1：关键转录因子活性热图。 颜色表示 ULM 活性分数（红色=激活，蓝色=抑制）。几个亮点：

SPI1（PU.1）：在巨噬细胞和单核细胞中高活性——这是髓系发育的"主控"TF，控制着巨噬细胞分化的整个基因表达程序。
HNF4A：只在肝细胞中高活性——这是肝细胞身份的核心 TF。如果你的肝细胞注释正确，HNF4A 应该是最特异的信号之一。
STAT1：在多种免疫细胞中活跃，尤其是巨噬细胞——STAT1 是干扰素信号通路的核心 TF。
FOXP3：只在 T 细胞亚群中有信号——这是调节性 T 细胞（Treg）的标志 TF。

💡 TF 活性热图是细胞注释的"另一种验证"。如果你的注释正确，那么 SPI1 应该在髓系细胞中最高、HNF4A 应该在肝细胞中最高。如果这些 TF 的活性模式和你的注释不一致——要么注释有问题，要么 ULM 推断不准（需要检查 CollecTRI 网络中该 TF 的靶基因覆盖度）。

图 2：关键转录因子在 UMAP 上的活性分布

## Part B：差异 TF 活性分析

### 为什么要做差异 TF 活性

Part A 告诉我们每种细胞类型的 TF 活性"基线"。但我们更关心的是：同一种细胞类型中，哪些 TF 在肝硬化中被激活或抑制了？

这比差异表达分析更有机制洞察：差异表达告诉你"有 500 个基因上调了"，而差异 TF 活性告诉你"这 500 个基因上调是因为 STAT1 被激活了"。

### Mann-Whitney U 检验

对每种细胞类型的每个 TF，比较 Healthy vs Cirrhotic 的活性分数：

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import \
    multipletests

results = []
cell_types = adata.obs["cell_type"].unique()

for ct in cell_types:
    mask = adata.obs["cell_type"] == ct
    h_mask = mask & (adata.obs["condition"] == "Healthy")
    c_mask = mask & (adata.obs["condition"] == "Cirrhotic")

    scores = adata.obsm["score_ulm"]
    for tf in scores.columns:
        h_vals = scores.loc[h_mask, tf].dropna()
        c_vals = scores.loc[c_mask, tf].dropna()
        if len(h_vals) < 10 or len(c_vals) < 10:
            continue
        stat, pval = mannwhitneyu(
            c_vals, h_vals, alternative="two-sided"
        )
        results.append({
            "cell_type": ct, "tf": tf,
            "mean_healthy": h_vals.mean(),
            "mean_cirrhotic": c_vals.mean(),
            "diff": c_vals.mean() - h_vals.mean(),
            "pval": pval
        })

diff_tf = pd.DataFrame(results)

### 多重检验校正

In [ ]:
diff_tf["padj"] = multipletests(
    diff_tf["pval"], method="fdr_bh"
)[1]

sig_tf = diff_tf[diff_tf["padj"] 0).sum()}")
print(f"  下调 (Cirrhotic): "
      f"{(sig_tf['diff']<0).sum()}")

**输出：**

> 📊 输出：
> 总测试数: 9012
> 显著差异 TF: 3847
>   上调 (Cirrhotic): 2103
>   下调 (Cirrhotic): 1744

9,012 次测试（12 种细胞类型 × ~750 个有足够细胞数的 TF），其中 3,847 个通过了 FDR < 0.05 的阈值。肝硬化中上调的 TF（2,103）略多于下调的（1,744），提示疾病状态下整体的转录调控网络更加活跃。

### 保存结果

In [ ]:
diff_tf.to_csv(
    "results/tables/tf_diff_condition.csv",
    index=False
)
print(f"✅ 已保存: tf_diff_condition.csv"
      f" ({len(diff_tf)} 行)")

### 关键发现

In [ ]:
# 每种细胞类型中差异最大的 TF
for ct in sorted(cell_types):
    sub = sig_tf[sig_tf["cell_type"] == ct]
    if len(sub) == 0:
        continue
    top_up = sub.nlargest(3, "diff")
    top_dn = sub.nsmallest(3, "diff")
    print(f"\n{ct}:")
    for _, r in top_up.iterrows():
        print(f"  ↑ {r['tf']}: diff={r['diff']:.3f}"
              f" (padj={r['padj']:.1e})")
    for _, r in top_dn.iterrows():
        print(f"  ↓ {r['tf']}: diff={r['diff']:.3f}"
              f" (padj={r['padj']:.1e})")

**输出：**

> 📊 输出：
> Macrophages:
>   ↑ STAT1: diff=1.847 (padj=2.3e-89)
>   ↑ IRF1: diff=1.562 (padj=8.1e-72)
>   ↑ NFKB1: diff=1.203 (padj=3.4e-45)
>   ↓ MAF: diff=-0.982 (padj=1.2e-31)
>   ↓ MAFB: diff=-0.876 (padj=4.5e-28)
>   ↓ NR1H3: diff=-0.654 (padj=2.8e-19)
> 
> Hepatocytes:
>   ↑ STAT3: diff=1.124 (padj=5.6e-34)
>   ↑ CEBPB: diff=0.893 (padj=1.2e-22)
>   ↑ JUN: diff=0.781 (padj=3.9e-18)
>   ↓ HNF4A: diff=-1.356 (padj=6.7e-51)
>   ↓ HNF1A: diff=-1.102 (padj=2.1e-38)
>   ↓ CEBPA: diff=-0.845 (padj=7.3e-24)
> 
> T cells:
>   ↑ STAT1: diff=0.934 (padj=1.8e-67)
>   ↑ TBX21: diff=0.612 (padj=4.2e-29)
>   ↑ IRF1: diff=0.578 (padj=2.7e-25)
>   ↓ TCF7: diff=-0.723 (padj=8.9e-34)
>   ↓ LEF1: diff=-0.689 (padj=5.1e-30)
>   ↓ FOXP3: diff=-0.234 (padj=3.2e-08)
> ...

这些结果讲述了一个完整的生物学故事：

巨噬细胞：STAT1/IRF1 上调 + MAF/MAFB 下调——这正是从"组织驻留型"（MAF/MAFB 高）向"炎症型"（STAT1/IRF1 高）巨噬细胞转变的分子特征。STAT1 驱动干扰素应答，IRF1 是其下游核心 TF。而 MAF/MAFB 是 Kupffer 细胞等组织驻留巨噬细胞的维持因子——它们的下调意味着 Kupffer 细胞的身份丢失。

肝细胞：HNF4A/HNF1A 下调 + STAT3/JUN 上调——HNF4A 是肝细胞分化和功能维持的核心 TF。它的活性下降意味着肝细胞正在"去分化"（dedifferentiation），这是肝硬化中肝功能丧失的分子基础。STAT3 和 JUN 的上调则反映了肝细胞在炎症微环境中的应激反应。

T 细胞：STAT1 上调 + TCF7/LEF1 下调——TCF7 和 LEF1 是 naive/memory T 细胞的维持因子。它们的下调提示 T 细胞从"静息态"转向"效应态"，与肝硬化中的慢性免疫激活一致。

### 差异 TF 活性可视化

In [ ]:
# 选择最有意义的 TF
plot_tfs = ["STAT1", "HNF4A", "SPI1",
            "FOXP3", "IRF1", "MAF"]
plot_cts = ["Macrophages", "Hepatocytes",
            "T cells", "NK cells"]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, tf in enumerate(plot_tfs):
    ax = axes[idx // 3, idx % 3]
    sub = diff_tf[
        (diff_tf["tf"] == tf) &
        (diff_tf["cell_type"].isin(plot_cts))
    ]
    x = range(len(sub))
    colors = ["#d62728" if d > 0 else "#1f77b4"
              for d in sub["diff"]]
    ax.bar(x, sub["diff"], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["cell_type"], rotation=45,
                        ha="right", fontsize=8)
    ax.set_title(tf, fontweight="bold")
    ax.set_ylabel("Δ TF activity")
    ax.axhline(0, color="gray", lw=0.5)

plt.suptitle(
    "Differential TF Activity (Cirrhotic - Healthy)",
    fontsize=14
)
plt.tight_layout()
plt.savefig(
    "results/figures/08_diff_tf_activity.png",
    dpi=150, bbox_inches="tight"
)
plt.close()

图 3：疾病 vs 健康差异转录因子热图

图 2：差异 TF 活性条形图。 红色表示肝硬化中活性上升，蓝色表示下降。注意 HNF4A 在肝细胞中的显著下调和 STAT1 在几乎所有免疫细胞中的上调。

## Part C：pySCENIC——更完整但更重的方案

### 什么是 SCENIC

SCENIC（Aibar et al., Nature Methods, 2017）是 TF 调控网络推断的"重型武器"。和 decoupler 使用现有 TF-靶基因数据库不同，SCENIC 直接从你的数据中从头推断调控关系：

GRNBoost2/GENIE3：用梯度提升回归从表达矩阵推断 TF-基因共表达关系
RcisTarget：用顺式调控元件（motif）数据库验证这些关系——只保留 TF 结合位点确实存在于靶基因启动子区域的调控关系
AUCell：对每个细胞计算 regulon（TF + 其验证后的靶基因集）的活性分数

### 为什么我们没有跑 SCENIC

坦白说，pySCENIC 在实操中有几个显著的障碍：

1. 数据库体积巨大。 pySCENIC 需要下载 motif 排名数据库（hg38__refseq-r80__10kb_up_and_down_tss.mc9nr.genes_vs_motifs.rankings.feather），单个文件约 20 GB。在国内网络环境下，下载常常失败。

2. 计算资源要求高。 对 58,735 个细胞运行 GRNBoost2，需要至少 64 GB 内存和数小时的运算时间。

3. 结果和 decoupler 高度互补。 SCENIC 的优势是"从头推断"——不依赖先验数据库。但对于已有高质量 TF-靶基因数据库（如 CollecTRI）的人类数据，decoupler 的结果已经足够好，而且速度快 100 倍。

### pySCENIC 代码框架（供参考）

如果你有资源和需求，以下是 pySCENIC 的核心代码框架：

In [ ]:
# === pySCENIC 代码框架（未实际运行）===
# 需要下载的数据库文件：
#   1. 排名数据库 (.feather, ~20GB)
#   2. Motif 注释 (.tbl)
# 下载地址: https://resources.aertslab.org/cistarget/

from pyscenic.utils import modules_from_adjacencies
from pyscenic.prune import prune2df, df2regulons
from pyscenic.aucell import aucell
from arboreto.algo import grnboost2

# Step 1: GRNBoost2 推断共表达
adjacencies = grnboost2(
    expression_data=adata.to_df(),
    tf_names=tf_list,
    seed=42
)

# Step 2: RcisTarget motif 验证
modules = modules_from_adjacencies(adjacencies, ...)
df_motifs = prune2df(adata, modules,
                     rankings_db, motif_anno)
regulons = df2regulons(df_motifs)

# Step 3: AUCell 活性评分
auc_mtx = aucell(adata, regulons)

💡 实际建议：对于大多数 scRNA-seq 项目，decoupler + CollecTRI 已经足够。只有当你需要发现新的、不在现有数据库中的调控关系时（比如非模式生物、或研究罕见的 TF），才需要上 SCENIC。

---

### 🔬 方法选择指南

🔬 方法选择指南：转录因子活性推断方法对比
方法核心原理速度需要什么输出适用场景decoupler + CollecTRI ✓线性模型(ULM)：从靶基因表达推断TF活性⚡ 秒级TF-靶基因数据库每个细胞的TF活性评分快速TF活性推断（推荐首选）pySCENICGRNBoost2(随机森林) → RcisTarget(motif验证) → AUCell(评分)🐢 小时级基因组注释+motif数据库(~4GB)调控子(regulon) + AUCell评分构建完整GRN；发现新调控关系SCENIC+联合RNA+ATAC → enhancer-TF-gene三元组🐢 小时级配对scRNA-seq + scATAC-seq增强子-TF-靶基因网络有多组学数据时CellOracleGRN + in silico TF扰动模拟中等基础GRN(SCENIC或ATAC推断)TF扰动后的模拟表达变化预测TF敲除/过表达效果
我们的选择：decoupler + CollecTRI。理由：

① 速度：整个分析在秒级完成，适合教学场景的快速迭代；
② 原理透明：ULM本质上是线性回归——如果TF的已知靶基因整体上调，则推断该TF活性升高。简单但有效；
③ CollecTRI数据库质量高：包含42,990个TF-靶基因关系，来源于文献整合，比纯motif预测更可靠。

何时应该用 pySCENIC

• 你想发现新的调控关系——decoupler依赖已知数据库，pySCENIC从数据中从头推断GRN
• 你想得到regulon（调控子）——pySCENIC输出的是"TF+其靶基因集合"，可以直接做下游分析
• 你有足够的计算资源——pySCENIC对10万细胞可能需要2-8小时（建议用GPU版本）

核心差异总结：decoupler回答"已知TF在不同细胞中活性如何"，pySCENIC回答"哪些TF可能调控了哪些基因"。前者是假设验证（hypothesis-driven），后者是假设生成（hypothesis-generating）。根据你的问题选择。

---

## 📖 与原文比较

Ramachandran et al., 2019 没有做系统的 TF 活性推断分析。他们主要通过差异表达和已知文献来讨论转录调控。但我们的分析结果和他们的生物学结论高度吻合：

巨噬细胞极化：原文描述的 SAMac 表达 TREM2/SPP1/GPNMB 等基因，这些基因受 SPI1 和 CEBPB 等髓系 TF 调控。我们检测到巨噬细胞中 SPI1 高活性 + STAT1/IRF1 在肝硬化中上调，为原文的"炎症巨噬细胞扩增"提供了调控层面的解释。

肝细胞去分化：原文提到肝硬化中肝细胞功能受损。我们发现 HNF4A（肝细胞核心 TF）活性在肝硬化中显著下调——这直接解释了为什么 ALB、CYP3A4 等肝细胞功能基因在疾病中表达降低。

纤维化信号：原文强调 TGF-β 信号在纤维化中的核心作用。我们检测到 SMAD3（TGF-β 的下游 TF）在星状细胞/间充质细胞中活性上升，和上一篇的通讯分析（TGFB1→TGFBR2）形成了完美的闭环：巨噬细胞分泌 TGFB1 → 星状细胞的 TGFBR2 接收 → SMAD3 被激活 → 纤维化基因被转录。

技术对比：我们使用了 decoupler 2.x + CollecTRI（ULM 方法），这在原文发表时尚不存在。CollecTRI 的 TF-靶基因覆盖度远超原文可能使用的任何数据库，使得 TF 活性推断的准确性和覆盖率大幅提升。

## 方法论反思

### TF 活性推断的局限

1. 数据库偏差。 CollecTRI 的 42,990 个交互主要来自免疫和癌症领域的研究。一些组织特异性的 TF-靶基因关系可能缺失——比如肝脏特异的 TF 网络可能覆盖不足。

2. "足迹"不等于"因果"。 ULM 检测到的是 TF 靶基因的协调变化（TF "足迹"），但这不能证明这个 TF 真的是驱动因子。可能是一个上游信号同时激活了这个 TF 和它的靶基因。

3. scRNA-seq 的稀疏性。 单细胞数据的 dropout 率很高（很多基因在某个细胞中检测为 0，但实际上是表达的）。这会影响 ULM 的准确性，尤其是对靶基因数量少的 TF。

4. 静态快照。 scRNA-seq 是一个时间点的快照。TF 的激活和靶基因的表达之间有时间滞后——一个刚被激活的 TF 可能还没来得及改变靶基因的 mRNA 水平。

💡 面对这些局限，最佳策略是：用 TF 活性分析生成候选假设，然后用正交实验验证。比如 ChIP-seq 验证 TF 结合、CRISPR KO 验证因果关系、蛋白质组学验证翻译后修饰状态。

## 输出文件清单

文件
内容
维度/行数

tf_activity_mean_celltype.csv
每种细胞类型的平均 TF 活性
12 × 1,185

tf_diff_condition.csv
差异 TF 活性（H vs C，per cell type）
9,012 行

08_tf_activity_heatmap.png
TF 活性热图
—

08_diff_tf_activity.png
差异 TF 活性条形图
—

## 小结

这一篇我们完成了：

✅ CollecTRI 网络加载（42,990 交互，1,185 TFs）
✅ decoupler 2.x ULM 方法推断全数据集 TF 活性
✅ 关键 TF 的细胞类型特异性验证：SPI1（髓系）、HNF4A（肝细胞）、FOXP3（Treg）
✅ 差异 TF 活性分析：9,012 次检验，3,847 个显著
✅ 核心生物学发现：巨噬细胞 STAT1/IRF1 激活 + MAF 抑制（炎症极化）；肝细胞 HNF4A 活性下降（去分化）
✅ pySCENIC 方法论介绍和实操建议

关键数字：

CollecTRI：42,990 个 TF-靶基因交互，1,185 个 TF
差异分析：9,012 次 Mann-Whitney 检验
显著差异 TF：3,847 个（FDR < 0.05）
核心 TF：SPI1、HNF4A、STAT1、FOXP3、IRF1、MAF

下一篇，我们将分析细胞组成的变化——肝硬化中哪些细胞类型的比例发生了显著改变？巨噬细胞亚群的此消彼长如何量化？这是从"分子层面"跨入"组织层面"的关键一步。